# Namma Space — Cloud 3D Gaussian Splatting Reconstruction Pipeline
**Techfest IIT Bombay 2026–27 | Round 1**

This notebook trains an authentic 3D Gaussian Splatting digital twin from your smartphone walkthrough video using an NVIDIA T4 GPU on Google Colab.

---
### Pre-Flight Checklist:
1. **Runtime Type**: Go to **Runtime → Change runtime type** and select **T4 GPU**.
2. **Recorded Video**: A 45–75 second MP4/MOV recorded following `docs/CAPTURE_SOP.md` (1080p landscape, locked AE/AF, dual-loop).

### Step 1: Verify NVIDIA GPU & CUDA

In [ ]:
!nvidia-smi

### Step 2: Clone Repositories & Install Dependencies (PyColmap + Inria 3DGS CUDA Rasterizer)

In [ ]:
import os, sys

# 1. Clone Roomsplat pipeline orchestrator
%cd /content
!git clone https://github.com/Ahmedn1/roomsplat.git
%cd /content/roomsplat
!pip install -e . pycolmap typer rich

# Ensure environment paths
sys.path.insert(0, "/content/roomsplat")
os.environ["PATH"] = "/root/.local/bin:/usr/local/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONPATH"] = "/content/roomsplat"
os.environ["ROOMSPLAT_GS_DIR"] = "/content/gaussian-splatting"

# 2. Clone official Inria 3D Gaussian Splatting engine
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
%cd /content/gaussian-splatting
!pip install submodules/diff-gaussian-rasterization
!pip install submodules/simple-knn

# 3. Configure Gaussian Splatting directory
%cd /content
!python3 -c "from roomsplat.cli import app; app()" config --gs-dir /content/gaussian-splatting

print("\nSUCCESS: Environment & CUDA rasterizers compiled successfully!")

### Step 3: Upload Your Smartphone Walkthrough Video

In [ ]:
from google.colab import files
import os

%cd /content
print("Upload your smartphone video (e.g. my_room.mp4):")
uploaded = files.upload()
if uploaded:
    video_filename = list(uploaded.keys())[0]
    print(f"\nUploaded video: {video_filename} ({os.path.getsize(video_filename) / (1024*1024):.2f} MB)")
else:
    print("No file uploaded via dialog. Checking for existing videos in /content...")

### Step 4: Run End-to-End Reconstruction (Extraction → SfM → 3DGS → Splat Export)

In [ ]:
%cd /content
import os, glob, subprocess

# Auto-detect video if not already defined
if 'video_filename' not in locals() or not os.path.exists(video_filename):
    vids = [f for f in glob.glob("/content/*") if f.lower().endswith(('.mp4', '.mov', '.m4v')) and not f.startswith('/content/sample_data')]
    if vids:
        video_filename = vids[0]
        print(f"Auto-detected uploaded video: {video_filename}")
    else:
        raise FileNotFoundError("No video found! Please run Step 3 to upload your video first.")

print(f"\nStarting 3D reconstruction on: {video_filename}")
print("Settings: 30,000 iterations, 1600px max-dim, low-VRAM mode on T4 GPU\n")

cmd = [
    "python3", "-c", "from roomsplat.cli import app; app()", "run",
    video_filename,
    "--name", "Namma Space Authentic Room",
    "--fps", "1.5",
    "--max-dim", "1600",
    "--iterations", "30000",
    "--low-vram",
    "--no-view"
]

env = dict(os.environ)
env["PYTHONPATH"] = "/content/roomsplat"
env["ROOMSPLAT_GS_DIR"] = "/content/gaussian-splatting"

res = subprocess.run(cmd, env=env)
if res.returncode == 0:
    print("\nSUCCESS: 3D Gaussian Splatting reconstruction finished!")
else:
    raise RuntimeError(f"Reconstruction failed with exit code {res.returncode}. Please check the log messages above.")

### Step 5: Download `scene.splat` and `cameras.json` for Namma Space Viewer

In [ ]:
import glob
import shutil
import os
from google.colab import files

# Find output directory
splat_dirs = glob.glob("/content/*_splat")
if splat_dirs:
    out_dir = splat_dirs[0]
    print(f"Found reconstruction output at: {out_dir}")
    
    splat_file = f"{out_dir}/scene.splat"
    cameras_file = f"{out_dir}/cameras.json"
    
    if os.path.exists(splat_file) and os.path.exists(cameras_file):
        shutil.copy(splat_file, "/content/scene.splat")
        shutil.copy(cameras_file, "/content/cameras.json")
        
        # Zip together for easy download
        !zip -j /content/namma_space_assets.zip /content/scene.splat /content/cameras.json
        
        print("\nDownloading namma_space_assets.zip to your computer...")
        files.download("/content/namma_space_assets.zip")
    else:
        print(f"Error: scene.splat or cameras.json not found in {out_dir}!")
else:
    print("Error: Output splat directory not found! Did Step 4 complete successfully?")